In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import torch

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    batch_size_train = 256
    batch_size_infer = 256
    n_epochs = 10
    max_len = 10
    model_name = "target_action"
    hidden_dim = 32
    embedding_dim = 64


config = Config()

In [4]:
print(utils.recall_vals_one_row([1, 2, 3], 2))
print(utils.recall_vals_one_row([1, 2, 3], 4))

True
False


In [5]:
utils.recall_several_targets(candidates=[[1, 2, 3], [1]], targets=[[1, 3, 2], [1, 2]])

0.8

In [6]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=16, progress_bar=True, use_memory_fs=False)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [7]:
%%time

PATH_DATA = '../data/02_intermediate/'
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
# df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
df_train = df_train.to_pandas()
# df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

df_train.name = "df_train"
# df_test.name = "df_test"

df_train = utils.make_date_stats(df_train)
# df_test = utils.make_date_stats(df_test)

df_train.drop(['date_day'], axis=1, inplace=True)
# df_test.drop(['date_day'], axis=1, inplace=True)

df_train = pl.from_pandas(df_train)
# df_test = pl.from_pandas(df_test)

users_with_carts_or_orders_train = (
    df_train.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select(config.col_user)
    .unique()
)

df_train = df_train.filter(
    pl.col(config.col_user).is_in(
        list(users_with_carts_or_orders_train.to_pandas()[config.col_user])
    )
)

df_train = df_train.with_columns(
    [(pl.col(config.col_user) + "_" + pl.col(config.col_n_week)).alias("user_week")]
)

users_weeks_with_carts_or_orders_train = (
    df_train.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select("user_week")
    .unique()
)

df_train = df_train.filter(
    pl.col("user_week").is_in(
        list(users_weeks_with_carts_or_orders_train.to_pandas()["user_week"])
    )
)

CPU times: user 3min 36s, sys: 1min 38s, total: 5min 14s
Wall time: 2min 38s


In [8]:
%%time

PATH_DATA = '../data/02_intermediate/'
# df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
# df_train = df_train.to_pandas()
df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

# df_train.name = "df_train"
df_test.name = "df_test"

# df_train = utils.make_date_stats(df_train)
df_test = utils.make_date_stats(df_test)

# df_train.drop(['date_day'], axis=1, inplace=True)
df_test.drop(['date_day'], axis=1, inplace=True)

# df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

users_with_carts_or_orders_test = (
    df_test.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select(config.col_user)
    .unique()
)

df_test = df_test.filter(
    pl.col(config.col_user).is_in(
        list(users_with_carts_or_orders_test.to_pandas()[config.col_user])
    )
)

df_test = df_test.with_columns(
    [(pl.col(config.col_user) + "_" + pl.col(config.col_n_week)).alias("user_week")]
)

users_weeks_with_carts_or_orders_test = (
    df_test.filter(
        (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
    )
    .select("user_week")
    .unique()
)

df_test = df_test.filter(
    pl.col("user_week").is_in(
        list(users_weeks_with_carts_or_orders_test.to_pandas()["user_week"])
    )
)

CPU times: user 5.44 s, sys: 1.27 s, total: 6.71 s
Wall time: 4.83 s


In [9]:
df_test.head()

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,user_week
i64,i64,i64,str,i64,datetime[ns],i8,i8,i8,i8,str
12899781,141736,1661724000559,"""clicks""",0,2022-08-29 00:00:00.559000064,29,35,0,0,"""12899781_35"""
12899781,199008,1661724022851,"""clicks""",0,2022-08-29 00:00:22.851000064,29,35,0,0,"""12899781_35"""
12899781,57315,1661724170835,"""clicks""",0,2022-08-29 00:02:50.835000064,29,35,0,0,"""12899781_35"""
12899781,194067,1661724246188,"""clicks""",0,2022-08-29 00:04:06.188,29,35,0,0,"""12899781_35"""
12899781,199008,1661780623778,"""clicks""",1,2022-08-29 15:43:43.777999872,29,35,0,15,"""12899781_35"""


In [10]:
# %%time

# users_with_carts_or_orders_train = (
#     df_train.filter(
#         (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
#     )
#     .select(config.col_user)
#     .unique()
# )

# users_with_carts_or_orders_test = (
#     df_test.filter(
#         (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
#     )
#     .select(config.col_user)
#     .unique()
# )

# df_train = df_train.filter(
#     pl.col(config.col_user).is_in(
#         list(users_with_carts_or_orders_train.to_pandas()[config.col_user])
#     )
# )

# df_test = df_test.filter(
#     pl.col(config.col_user).is_in(
#         list(users_with_carts_or_orders_test.to_pandas()[config.col_user])
#     )
# )

In [11]:
# %%time

# df_train = df_train.with_columns(
#     [(pl.col(config.col_user) + "_" + pl.col(config.col_n_week)).alias("user_week")]
# )

# df_test = df_test.with_columns(
#     [(pl.col(config.col_user) + "_" + pl.col(config.col_n_week)).alias("user_week")]
# )

# users_weeks_with_carts_or_orders_train = (
#     df_train.filter(
#         (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
#     )
#     .select("user_week")
#     .unique()
# )

# users_weeks_with_carts_or_orders_test = (
#     df_test.filter(
#         (pl.col(config.col_type) == "carts") | (pl.col(config.col_type) == "orders")
#     )
#     .select("user_week")
#     .unique()
# )

# df_train = df_train.filter(
#     pl.col("user_week").is_in(
#         list(users_weeks_with_carts_or_orders_train.to_pandas()["user_week"])
#     )
# )

# df_test = df_test.filter(
#     pl.col("user_week").is_in(
#         list(users_weeks_with_carts_or_orders_test.to_pandas()["user_week"])
#     )
# )

In [12]:
%%time

# test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_type_group] = df[config.col_type].apply(lambda x: 'clicks' if x == 'clicks' else 'target_action')
df[config.col_num_action + '_per_type_group'] = df.groupby([config.col_user, config.col_type_group])[config.col_aid].cumcount()
df[config.col_num_action] = df.groupby([config.col_user])[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int16, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int16, strict=True)
        .alias(config.col_n_session),
    ]
)

CPU times: user 5min 6s, sys: 1min 5s, total: 6min 12s
Wall time: 1min 52s


In [13]:
del (
    df_train,
    df_test,
    users_with_carts_or_orders_train,
    users_with_carts_or_orders_test,
    users_weeks_with_carts_or_orders_train,
    users_weeks_with_carts_or_orders_test,
)
gc.collect()

256

In [14]:
# ### IF DEBUG PIPELINE !!!

# df = df.filter(pl.col("user") % 100 == 0)

In [15]:
counts_aid = df.groupby([config.col_aid]).count().sort("count", reverse=True)

In [16]:
df.head()

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,user_week,type_group,num_action_per_type_group,num_action
i64,i64,i64,str,i16,datetime[ns],i8,i8,i8,i8,str,str,i64,i16
13629931,1576043,1662328770401,"""clicks""",1,2022-09-04 23:59:30.400999936,4,35,6,23,"""13629931_35""","""clicks""",0,0
14568911,231093,1662328768077,"""clicks""",0,2022-09-04 23:59:28.076999936,4,35,6,23,"""14568911_35""","""clicks""",0,0
14250687,374291,1662328764970,"""carts""",2,2022-09-04 23:59:24.969999872,4,35,6,23,"""14250687_35""","""target_action""",0,0
14362537,1142000,1662328764079,"""clicks""",1,2022-09-04 23:59:24.079000064,4,35,6,23,"""14362537_35""","""clicks""",0,0
13723496,1240764,1662328761140,"""carts""",1,2022-09-04 23:59:21.140,4,35,6,23,"""13723496_35""","""target_action""",0,0


In [17]:
counts_aid.head()

aid,count
i64,u32
485256,109874
29735,68351
1733943,64835
832192,56178
1460571,55084


In [18]:
df.shape

(126764258, 14)

In [19]:
# df = df.to_pandas()

In [20]:
# def split_data(df, validation_cut, test_users):
#     df_val = df[
#         (df[config.col_num_action] <= validation_cut)
#         & (df[config.col_user].isin(test_users))
#     ]

#     df_val_seq = df[
#         (df[config.col_num_action] > validation_cut)
#         & (df[config.col_user].isin(test_users))
#     ]

#     df_train = df.drop(df_val.index)
#     return df_train, df_val, df_val_seq

In [21]:
# %%time

# X_train, X_val, X_val_seq = split_data(df, validation_cut=2, test_users=test_users)

In [22]:
%%time

trs = 40
counts_aid_filtered = counts_aid.filter(pl.col("count") >= trs).to_pandas()
pad = len(counts_aid_filtered)

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=pad)
ordinal_encoder.fit(counts_aid_filtered[config.col_aid].values.reshape(-1, 1))

CPU times: user 257 ms, sys: 3.1 ms, total: 260 ms
Wall time: 258 ms


OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=413313)

In [23]:
del df, counts_aid, counts_aid_filtered
gc.collect()

0

In [24]:
class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super(Attention, self).__init__()

        self.hidden_size = hidden_size
        self.batch_first = batch_first

        self.att_weights = nn.Parameter(
            torch.Tensor(1, hidden_size), requires_grad=True
        )

        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def get_mask(self):
        pass

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        # print(f"inputs: {inputs.size()}")

        # apply attention layer
        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0)  # (1, hidden_size) --> (hidden_size, 1)
            .unsqueeze(0)  # (1, hidden_size, 1)
            .repeat(batch_size, 1, 1),  # (batch_size, hidden_size, 1)
        )

        # print(f"weights: {weights.size()}")

        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)
        # print(f"attentions: {attentions.size()}")

        # create mask based on the sentence lengths
        mask = torch.ones(attentions.size(), requires_grad=True).to(device)
        # print(f"mask: {mask.size()}")
        if batch_size == 1:
            mask = mask.view(1, -1)
        for i, l in enumerate(lengths):  # skip the first sentence
            if l < max_len:
                mask[i, l:] = 0

        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        _sums = masked.sum(-1).unsqueeze(-1)  # sums per row

        attentions = masked.div(_sums)

        # apply attention weights
        weighted = torch.mul(inputs, attentions.unsqueeze(-1).expand_as(inputs))
        # print(f"weighted: {weighted.size()}")

        # get the final fixed vector representations of the sentences
        representations = weighted.sum(1).squeeze()
        # print(f"representations: {representations.size()}")

        return representations, attentions

In [25]:
n_vocab = pad + 1

In [26]:
class RRNN(nn.Module):
    def __init__(
        self,
        n_vocab=n_vocab,
        hidden_dim=config.hidden_dim,
        embedding_dim=config.embedding_dim,
        dropout=0.1,
    ):
        super(RRNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=False)
        self.hidden_dim = hidden_dim
        self.out_features_num = n_vocab
        self.dropout0 = nn.Dropout(dropout, inplace=True)
        self.lstm_desc = nn.GRU(
            embedding_dim,
            hidden_dim,
            dropout=dropout,
            num_layers=1,
            bidirectional=False,
            batch_first=True,
        )

        self.pooling_result_dimension = self.hidden_dim * 1
        self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

        self.attention_descs = Attention(self.hidden_dim * 1, batch_first=True)
        self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
        self.dropout1 = nn.Dropout(dropout, inplace=True)
        self.fc1 = nn.Linear(self.dim_after_concat, self.embedding_dim)
        self.dropout2 = nn.Dropout(dropout, inplace=True)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(self.embedding_dim, self.out_features_num)

        self.fc2.weight = self.embeddings.weight

        # self.category_embedding = self._create_embedding_projection(cardinality=10, embed_size=6, padding_idx=None, scale_grad_by_freq=False)
        # self.subcategory_embedding = self._create_embedding_projection(cardinality=50, embed_size=14, padding_idx=None, scale_grad_by_freq=False)
        # self.region_embedding = self._create_embedding_projection(cardinality=85, embed_size=19, padding_idx=None, scale_grad_by_freq=False)

    def _create_embedding_projection(
        self,
        cardinality,
        embed_size,
        add_missing=True,
        padding_idx=0,
        scale_grad_by_freq=False,
    ):
        add_missing = add_missing * 1
        return nn.Embedding(
            num_embeddings=cardinality + add_missing,
            embedding_dim=embed_size,
            padding_idx=padding_idx,
            scale_grad_by_freq=scale_grad_by_freq,
        )

    def forward(self, aids, lens):
        batch_size = aids.shape[0]

        embedded_aids = self.embeddings(aids)
        embedded_aids = self.dropout0(embedded_aids)
        pad_embed_pack = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_aids, list(lens), batch_first=True, enforce_sorted=False
        )
        states_aids, hh_aids = self.lstm_desc(pad_embed_pack)

        states_aids, lens_unpacked = torch.nn.utils.rnn.pad_packed_sequence(
            states_aids, batch_first=True
        )

        lengths_aids = [x_.size(1) for x_ in states_aids]
        atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
        atten_aids = atten_aids.view(batch_size, -1)

        features = torch.cat(
            [
                # subcategory_embed, category_embed, region_embed,
                atten_aids
            ],
            dim=-1,
        )
        features = self.bn1(features)
        features = self.dropout1(features)
        features = features.view(features.size(0), -1)
        features = self.fc1(features)
        features = self.dropout2(features)
        features = self.relu1(features)
        features = self.fc2(features)
        # features = F.normalize(features)

        return features

In [27]:
df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")

In [28]:
df_test = df_test.sort(config.col_ts)

In [29]:
df_test = df_test.to_pandas()
df_test[config.col_aid] = ordinal_encoder.transform(
    df_test[config.col_aid].values.reshape(-1, 1)
).astype(int)
df_test = pl.from_pandas(df_test)

In [30]:
df_test

user,aid,ts,type,n_session
i64,i64,i64,str,i64
12899779,413313,1661724000278,"""clicks""",0
12899780,254037,1661724000378,"""clicks""",0
12899781,31722,1661724000559,"""clicks""",0
12899782,371622,1661724000568,"""clicks""",0
12899783,56739,1661724000572,"""clicks""",0
12899784,230462,1661724000604,"""clicks""",0
12899785,413313,1661724000809,"""clicks""",0
12899786,413313,1661724001174,"""clicks""",0
12899787,374685,1661724001617,"""clicks""",0


In [31]:
%%time

df_test = (
    df_test.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list().alias(config.col_train_seq),
        ]
    )
)

CPU times: user 1.26 s, sys: 21 s, total: 22.2 s
Wall time: 644 ms


In [32]:
df_test = df_test.with_column(pl.col(config.col_train_seq).apply(len).alias("len"))

In [33]:
df_test

user,train_seq,len
i64,list[i64],i64
14454720,[234098],1
12932096,"[21766, 192373, 304352]",3
14142144,"[94671, 356534, ... 55717]",16
12942208,[238301],1
13347904,"[159443, 171219, ... 256694]",11
13879808,[274720],1
13527808,"[413313, 413313]",2
13909696,[6785],1
13213376,"[37189, 305746]",2


In [34]:
df_test[["len"]].describe()

describe,len
str,f64
"""count""",1.671803e6
"""null_count""",0.0
"""mean""",4.144103
"""std""",8.215717
"""min""",1.0
"""max""",458.0
"""median""",2.0


In [35]:
%%time

df_test = df_test.to_pandas()

to_dataset = pd.DataFrame(df_test[config.col_train_seq].tolist(), index=df_test.index)
to_dataset.columns = ['pos' + '_' + str(pos) for pos in to_dataset.columns]
to_dataset[config.col_user] = df_test[config.col_user]

df_test = pl.from_pandas(df_test)

cols_pos = [col for col in to_dataset.columns if "pos" in col]

PATH_DATASET = 'to_dataset.parquet'

to_dataset.to_parquet(PATH_DATASET, index=False)

dataset = Dataset_merlin(PATH_DATASET)
dl_merlin = Loader(dataset=dataset, batch_size=config.batch_size_infer, shuffle=False)

del df_test, to_dataset
gc.collect()

import math


def get_matrix_and_target_from_batch(
    batch, cols_pos=cols_pos, cols_target_val=None, pad=pad, mod='inference'
):
    assert mod in ['inference', 'train', 'valid']

    batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
    for i in range(len(cols_pos)):
        batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
    batch_matrix = [
        torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
        for i in range(len(batch_matrix))
    ]
    lens = torch.tensor(list(map(len, batch_matrix)))

    batch_matrix = torch.nn.utils.rnn.pad_sequence(
        batch_matrix, batch_first=True, padding_value=pad
    )

    users = batch[config.col_user].squeeze(1).tolist()
    
    if mod == 'train':
        target = batch[config.col_target]
    elif mod == 'valid':
        target = torch.zeros([len(batch[cols_target_val[0]]), len(cols_target_val)])
        for i in range(len(cols_target_val)):
            target[:, i] = torch.reshape(batch[cols_target_val[i]], (1, -1))
        target = [
            torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
            for i in range(len(target))
        ]
    elif mod == 'inference':
        target = None
    return batch_matrix, target, lens, users

CPU times: user 51.6 s, sys: 28.9 s, total: 1min 20s
Wall time: 43.8 s


In [36]:
def inference_func(loader, model, num_candidates=20):
    model.eval()

    bar = tqdm(loader)
    list_preds = list()
    list_users = list()

    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, _, lens, users = get_matrix_and_target_from_batch(
                batch, mod="inference"
            )
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            logits = model(aids, lens)
            preds_cur = (
                torch.topk(logits, num_candidates).indices.detach().cpu().numpy()
            )
            list_preds.append(preds_cur)
            list_users.append(users)

    list_preds = utils.flatten_list(list_preds)
    list_users = utils.flatten_list(list_users)

    df_preds = pd.DataFrame({"session": list_users, "preds": list_preds})
    df_preds = df_preds.explode("preds")

    df_preds["preds"] = ordinal_encoder.inverse_transform(
        df_preds["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_preds = df_preds.dropna()
    df_preds["preds"] = df_preds["preds"].astype(int)
    df_preds = pl.from_pandas(df_preds)
    df_preds = (
        (
            df_preds.groupby(["session"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("session")
        .to_pandas()
    )

    # PREDS = df_preds["preds"].tolist()
    # PREDS = [list(pred) for pred in PREDS]

    return df_preds

In [37]:
model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{5 + 1}.pth",
    )
)

model.to(device)

/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.


/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.




RRNN(
  (embeddings): Embedding(413314, 64)
  (dropout0): Dropout(p=0.1, inplace=True)
  (lstm_desc): GRU(64, 32, batch_first=True, dropout=0.1)
  (attention_descs): Attention()
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.1, inplace=True)
  (fc1): Linear(in_features=32, out_features=64, bias=True)
  (dropout2): Dropout(p=0.1, inplace=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=64, out_features=413314, bias=True)
)

In [38]:
%%time

df_preds = inference_func(dl_merlin, model)

100%|███████████████████████████████████████| 6531/6531 [19:10<00:00,  5.68it/s]


CPU times: user 21min 33s, sys: 39.7 s, total: 22min 13s
Wall time: 19min 25s


In [39]:
# df_test.head()

In [40]:
df_preds["len"] = df_preds["preds"].progress_apply(len)

100%|████████████████████████████| 1671803/1671803 [00:01<00:00, 1211098.36it/s]


In [41]:
df_preds.head()

,session,preds,len
0,12899779,"[1006198, 1030727, 673407, 409620, 1758603, 12...",20
1,12899780,"[487136, 1758603, 1032776, 736515, 889686, 760...",20
2,12899781,"[918667, 606307, 759436, 3542, 199008, 1008624...",20
3,12899782,"[1714681, 1675581, 533068, 319580, 1627951, 13...",20
4,12899783,"[26203, 1083665, 1022566, 1305442, 1257293, 77...",20


In [42]:
sub = pd.read_csv("../data/01_raw/sample_submission.csv")
# test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
# test = test["user"].unique()

In [43]:
sub.head()

,session_type,labels
0,12899779_clicks,129004 126836 118524
1,12899779_carts,129004 126836 118524
2,12899779_orders,129004 126836 118524
3,12899780_clicks,129004 126836 118524
4,12899780_carts,129004 126836 118524


In [44]:
df_preds["session"] = list(map(str, df_preds["session"]))

In [45]:
%%time

dfs = list()
for session_type in ['clicks', 'carts', 'orders']:
    df_cur = df_preds[['session', "preds"]]
    df_cur.rename({"preds": 'labels', 'session': 'session_type'}, axis=1, inplace=True)
    df_cur['session_type']= df_cur['session_type'].progress_apply(lambda x: x + '_' + session_type)
    df_cur['labels']= df_cur['labels'].progress_apply(lambda x: ' '.join(str(aid) for aid in x))
    dfs.append(df_cur)

del df_cur
gc.collect()

pred_df = pd.concat(
    dfs
).reset_index(drop=True)

del dfs
gc.collect()

# pred_df = pl.from_pandas(pred_df)
# pred_df = pred_df.sort("session_type", reverse=False)

pred_df = sub[['session_type']].merge(pred_df, on=['session_type'])

100%|██████████████████████████████| 1671803/1671803 [00:26<00:00, 62761.84it/s]


CPU times: user 1min 30s, sys: 1.57 s, total: 1min 32s
Wall time: 1min 32s


In [46]:
pred_df.to_csv(
    f"../data/07_model_output/sub_{config.model_name}_epoch{5 + 1}.csv", index=False
)

In [47]:
pred_df.head()

,session_type,labels
0,12899779_clicks,1006198 1030727 673407 409620 1758603 1257293 ...
1,12899779_carts,1006198 1030727 673407 409620 1758603 1257293 ...
2,12899779_orders,1006198 1030727 673407 409620 1758603 1257293 ...
3,12899780_clicks,487136 1758603 1032776 736515 889686 760500 49...
4,12899780_carts,487136 1758603 1032776 736515 889686 760500 49...


In [48]:
stop

NameError: name 'stop' is not defined

In [ ]:
clicks_pred_df = df_preds[["preds"]].add_suffix(
    "_clicks"
)  # .to_frame("labels").reset_index()

In [ ]:
clicks_pred_df

In [ ]:
clicks_pred_df = df_preds.add_suffix("_clicks").to_frame('labels').reset_index()
orders_pred_df = df_preds.add_suffix("_orders").to_frame('labels').reset_index()
carts_pred_df = df_preds.add_suffix("_carts").to_frame('labels').reset_index()
pred_df = pd.concat(
    [clicks_pred_df, orders_pred_df, carts_pred_df]
)
pred_df.columns = ["session_type", "labels"]

In [ ]:
df_preds[["len"]].describe()

In [ ]:
stop

In [ ]:
df_test

In [ ]:
stop

In [ ]:
%%time

X_train[config.col_aid] = ordinal_encoder.transform(
    X_train[config.col_aid].values.reshape(-1, 1)
).astype(int)
X_val_seq[config.col_aid] = ordinal_encoder.transform(
    X_val_seq[config.col_aid].values.reshape(-1, 1)
).astype(int)

In [ ]:
%%time

X_train = pl.from_pandas(X_train)
X_train = X_train.sort(config.col_ts)

X_val = pl.from_pandas(X_val)
X_val = X_val.sort(config.col_ts)

X_val_seq = pl.from_pandas(X_val_seq)
X_val_seq = X_val_seq.sort(config.col_ts)

In [ ]:
X_train.filter(pl.col(config.col_user) == 0)

In [ ]:
%%time

X_train = (
    X_train.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
)

X_val = (
    X_val.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

X_val_seq = (
    X_val_seq.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

In [ ]:
X_train.shape

In [ ]:
X_train.filter(pl.col(config.col_user) == 0)

In [ ]:
# dict_map_targets = dict()
# dict_map_targets["clicks"] = 0
# dict_map_targets["carts"] = 1
# dict_map_targets["orders"] = 2

In [ ]:
counts_aid["aid"].min(), counts_aid["aid"].max()

In [ ]:
pad

In [ ]:
# %%time

# df_train = df_train.to_pandas()
# df_train['aid_filtered'] = ordinal_encoder.transform(df_train['aid'].values.reshape(-1, 1)).astype(int)
# df_train = pl.from_pandas(df_train)

In [ ]:
# counts_aid_type.filter((pl.col("type") == "orders") & (pl.col("count") >= 1)).select(
#     ["aid"]
# ).unique().shape

In [ ]:
# counts_aid_type

In [ ]:
counts_aid.tail()

In [ ]:
counts_aid.filter(pl.col("aid") == 1)

In [ ]:
# len(counts_aid), len(counts_aid_filtered)

In [ ]:
# %%time

# token_to_idx = dict()
# # token_to_idx[-1] = 0 # pad
# token_to_idx[0] = 0 # unknown / pad

# idx = 1
# for i in tqdm(range(len(counts_aid["aid"]))):
#     aid = counts_aid["aid"][i]
#     cnt = counts_aid["count"][i]
#     if cnt > 4:
#         token_to_idx[aid] = idx
#         idx += 1
#     # else:
#     #     token_to_idx[aid] = 0

# idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [ ]:
# len(token_to_idx)

In [ ]:
# min(list(token_to_idx.values()))

In [ ]:
# token_to_idx[0]

In [ ]:
# max_len = 10
# # pad = counts_aid["aid"].max() + 1

# # def get_id_of_token(token):
# #     if token in set(list(token_to_idx.keys())[1:]):
# #         idx = token_to_idx[token]
# #     else:
# #         idx = 0  # unknown
# #     return idx


# # def get_token_if_id(idx):
# #     if idx in set(list(idx_to_token.keys())[1:]):
# #         token = idx_to_token[idx]
# #     else:
# #         token = 0  # unknown
# #     return token


# def to_matrix(names, max_len=max_len, pad=pad, dtype="int32", batch_first=True):
#     """Casts a list of names into rnn-digestible matrix"""

#     max_len = max_len or max(map(len, names))
#     names_ix = np.zeros([len(names), max_len], dtype) + pad

#     for i in tqdm(range(len(names))):
#         cur_name = names[i]
#         line_ix = cur_name[-max_len:]  # last max_len elements
#         # cur_name = cur_name[-max_len:]  # last max_len elements
#         # line_ix = [
#         #     get_id_of_token(cur_name[c]) for c in range(len(cur_name)) if c < max_len
#         # ]
#         names_ix[i, -len(line_ix) :] = line_ix

#     if not batch_first:  # convert [batch, time] into [time, batch]
#         names_ix = np.transpose(names_ix)

#     return names_ix

In [ ]:
l = [1, 2, 3]
print(utils.split_list_randomly(l=l, seed=1))
print(utils.split_list_randomly(l=l, seed=2))

In [ ]:
X_train.head()

In [ ]:
X_train.shape

In [ ]:
X_val.head()

In [ ]:
%%time

X_train = X_train.to_pandas()
X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: x[:-1])
X_train[config.col_target] = X_train[config.col_aid].parallel_apply(lambda x: x[-1])
X_train = pl.from_pandas(X_train)

In [ ]:
X_train.head()

In [ ]:
# %%time

# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[:-1]).alias(config.col_train_seq)))
# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[-1]).alias(config.col_target)))

In [ ]:
# %%time

# X_train['val'] = X_train["val"].progress_apply(lambda x: x[0]) #orig_vals

# col_aids = 'train'
# X_train = pl.from_pandas(X_train)
# X_train = (
#     X_train.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_train = X_train[['user', 'train', 'val']]

In [ ]:
# %%time

# max_len = 10

# list_train = list()
# list_target = list()
# list_user = list()

# n_iter = 5
# for i in tqdm(range(len(X_train))):
#     row = X_train.iloc[i]
#     row_train = list(row['train'])
#     target_cur_main = row["val"]
#     len_row_train = len(row_train)
#     if len_row_train == 1 and target_cur_main < pad:
#         list_train.append(tuple(row_train))
#         list_target.append(target_cur_main)
#         list_user.append(row['user'])
#     else:
#         n_iter_cur = min(n_iter, len_row_train - 1)
#         train_cur_main = utils.truncate_list(l=row_train, max_len=max_len, from_start=False)
#         if target_cur_main < pad:
#             list_train.append(tuple(train_cur_main))
#             list_target.append(target_cur_main)
#             list_user.append(row['user'])

#         for seed in range(n_iter_cur):
#             train_cur, target_cur = utils.split_list_randomly(l=row_train, seed=seed)
#             target_cur = target_cur[0]
#             if target_cur < pad:
#                 train_cur = utils.truncate_list(l=train_cur, max_len=max_len, from_start=False)
#                 list_train.append(tuple(train_cur))
#                 list_target.append(target_cur)
#                 list_user.append(row['user'])

# X_train = pd.DataFrame()
# X_train['user'] = list_user
# X_train['train'] = list_train
# X_train['val'] = list_target
# X_train.drop_duplicates(inplace=True)
# X_train.reset_index(drop=True, inplace=True)

In [ ]:
%%time

X_train = X_train.to_pandas()
X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False))
X_train = pl.from_pandas(X_train)

In [ ]:
X_train.head()

In [ ]:
X_train = X_train.filter(pl.col(config.col_target) != pad)

In [ ]:
X_train[config.col_target].max(), pad  # (31207, 31211)

In [ ]:
X_train.shape

In [ ]:
X_train[config.col_target].value_counts()

In [ ]:
X_val_seq.head()

In [ ]:
X_val_seq = X_val_seq.to_pandas()
X_val_seq[config.col_train_seq] = X_val_seq[config.col_aid].parallel_apply(
    lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
)
X_val_seq = pl.from_pandas(X_val_seq)

In [ ]:
X_val_seq.head()

In [ ]:
X_val = X_val.join(
    X_val_seq[[config.col_user, config.col_train_seq]], how="inner", on=config.col_user
)

In [ ]:
X_val = X_val.rename({config.col_aid: config.col_target})

In [ ]:
X_val.filter(pl.col(config.col_user) == 13833088)

In [ ]:
# %%time

# X_val["train"] = X_val["train"].progress_apply(lambda x: utils.truncate_list(l=list(x), max_len=max_len, from_start=False))
# X_val["val"] = X_val["val"].progress_apply(lambda x: x[0]) #orig_vals

In [ ]:
# %%time

# X_val = X_val.explode('train')
# X_val['train'] = ordinal_encoder.transform(X_val['train'].values.reshape(-1, 1)).astype(int)

# col_aids = 'train'
# X_val = pl.from_pandas(X_val)
# X_val = (
#     X_val.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_val = X_val[['user', 'train', 'val']]

In [ ]:
# %%time

# df_train_clicks["val"] = df_train_clicks["val"].progress_apply(lambda x: x[0])

In [ ]:
# df_train_clicks["val"].value_counts()

In [ ]:
# df_train_clicks.head()

In [ ]:
pad

In [ ]:
# df_train[col_aids].value_counts().sort("counts", reverse=True)

In [ ]:
# df_train_clicks.drop([col_aids, "split"], axis=1, inplace=True)

In [ ]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [pl.col("aid").apply(lambda x: utils.split_list_randomly(list(x))).alias("split")]
# )

In [ ]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [
#         pl.col("split").apply(lambda x: x[0]).alias("train"),
#         pl.col("split").apply(lambda x: x[1]).alias("val"),
#     ]
# )

# for col in ["aid", "split"]:
#     df_train_clicks.drop_in_place(col)

# df_train_clicks = df_train_clicks.sort("user")

In [ ]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.to_pandas()

In [ ]:
# %%time

# df_train_clicks_mapped["val"] = df_train_clicks_mapped["val"].parallel_apply(
#     lambda x: get_id_of_token(x[0])
# )

In [ ]:
# shape_orig = df_train_clicks_mapped.shape
# df_train_clicks_mapped = df_train_clicks_mapped[
#     df_train_clicks_mapped["val"] != 0
# ].reset_index(drop=True)
# print(shape_orig, "->", df_train_clicks_mapped.shape)

In [ ]:
# (12773, 4)

In [ ]:
X_train.head()

In [ ]:
X_val.head()

In [ ]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].progress_apply(lambda x: utils.truncate_list(list(x), max_len=max_len))

In [ ]:
# %%time

# df_train_clicks["train"] = list(to_matrix(df_train_clicks["train"]))

In [ ]:
# df_train_clicks["val"].value_counts()

In [ ]:
# df_train_clicks.head()

In [ ]:
# df_train_clicks["train"][0]

In [ ]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].parallel_apply(
#     lambda x: to_matrix([x])[0]
# )

In [ ]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.with_columns(
#     [
#         pl.col("train").apply(lambda x: to_matrix(list(x))[0]).alias("train"),
#         pl.col("val").apply(lambda x: to_matrix(list('val')[:1])[0]).alias("val"),
#     ]
# )
# df_train_clicks_mapped = df_train_clicks_mapped.sort("user")

In [ ]:
# class Dataset_RNN(Dataset):
#     def __init__(self, df, mode):

#         self.df = df  # .reset_index(drop=True)
#         self.mode = mode

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, index):
#         row = self.df.iloc[index]
#         user = row["user"]
#         aids_train = row["train"]
#         # aids_train = to_matrix([aids_train])[0]
#         aids_val = row["val"]
#         # aids_val = to_matrix([aids_val])[0]

#         if self.mode == "test":
#             return torch.tensor(aids_train).long()
#         else:
#             return torch.tensor(aids_train).long(), torch.tensor(aids_val).float()

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super(Attention, self).__init__()

        self.hidden_size = hidden_size
        self.batch_first = batch_first

        self.att_weights = nn.Parameter(
            torch.Tensor(1, hidden_size), requires_grad=True
        )

        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def get_mask(self):
        pass

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        # print(f"inputs: {inputs.size()}")

        # apply attention layer
        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0)  # (1, hidden_size) --> (hidden_size, 1)
            .unsqueeze(0)  # (1, hidden_size, 1)
            .repeat(batch_size, 1, 1),  # (batch_size, hidden_size, 1)
        )

        # print(f"weights: {weights.size()}")

        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)
        # print(f"attentions: {attentions.size()}")

        # create mask based on the sentence lengths
        mask = torch.ones(attentions.size(), requires_grad=True).to(device)
        # print(f"mask: {mask.size()}")
        if batch_size == 1:
            mask = mask.view(1, -1)
        for i, l in enumerate(lengths):  # skip the first sentence
            if l < max_len:
                mask[i, l:] = 0

        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        _sums = masked.sum(-1).unsqueeze(-1)  # sums per row

        attentions = masked.div(_sums)

        # apply attention weights
        weighted = torch.mul(inputs, attentions.unsqueeze(-1).expand_as(inputs))
        # print(f"weighted: {weighted.size()}")

        # get the final fixed vector representations of the sentences
        representations = weighted.sum(1).squeeze()
        # print(f"representations: {representations.size()}")

        return representations, attentions

In [ ]:
# min(600, round(1.6 * n_cat**0.56))

n_vocab = pad + 1
hidden_dim = 128
embedding_dim = min(600, round(1.6 * pad**0.56))
embedding_dim = 100

In [ ]:
embedding_dim

In [ ]:
# class Simple_LSTM(nn.Module):
#     def __init__(
#         self,
#         out_features_num,
#         n_vocab=n_vocab,
#         hidden_dim=hidden_dim,
#         embedding_dim=embedding_dim,
#         dropout=0.1,
#     ):
#         super(Simple_LSTM, self).__init__()
#         self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=True)
#         self.hidden_dim = hidden_dim
#         self.out_features_num = out_features_num

#         self.lstm_desc = nn.LSTM(
#             embedding_dim,
#             hidden_dim,
#             dropout=dropout,
#             num_layers=2,
#             bidirectional=True,
#             batch_first=True,
#         )

#         self.pooling_result_dimension = self.hidden_dim * 2
#         self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

#         self.attention_descs = Attention(self.hidden_dim * 2, batch_first=True)
#         self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
#         self.dropout1 = nn.Dropout(dropout, inplace=True)
#         self.fc1 = nn.Linear(self.dim_after_concat, self.pooling_result_dimension)
#         self.dropout2 = nn.Dropout(dropout, inplace=True)
#         self.fc2 = nn.Linear(self.pooling_result_dimension, self.out_features_num)

#         # self.category_embedding = self._create_embedding_projection(cardinality=10, embed_size=6, padding_idx=None, scale_grad_by_freq=False)
#         # self.subcategory_embedding = self._create_embedding_projection(cardinality=50, embed_size=14, padding_idx=None, scale_grad_by_freq=False)
#         # self.region_embedding = self._create_embedding_projection(cardinality=85, embed_size=19, padding_idx=None, scale_grad_by_freq=False)

#     def _create_embedding_projection(
#         self,
#         cardinality,
#         embed_size,
#         add_missing=True,
#         padding_idx=0,
#         scale_grad_by_freq=False,
#     ):
#         add_missing = add_missing * 1
#         return nn.Embedding(
#             num_embeddings=cardinality + add_missing,
#             embedding_dim=embed_size,
#             padding_idx=padding_idx,
#             scale_grad_by_freq=scale_grad_by_freq,
#         )

#     def forward(self, aids):
#         batch_size = aids.shape[0]

#         # category_embed = self.category_embedding(category)
#         # subcategory_embed = self.subcategory_embedding(subcategory)
#         # region_embed = self.region_embedding(region)

#         embedded_aids = self.embeddings(aids)

#         states_aids, hh_aids = self.lstm_desc(embedded_aids)

#         lengths_aids = [x_.size(1) for x_ in states_aids]
#         atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
#         atten_aids = atten_aids.view(batch_size, -1)

#         features = torch.cat(
#             [
#                 # subcategory_embed, category_embed, region_embed,
#                 atten_aids
#             ],
#             dim=-1,
#         )
#         # features = self.bn1(features)
#         features = self.dropout1(features)
#         features = features.view(features.size(0), -1)
#         features = self.fc1(features)
#         features = self.dropout2(features)
#         features = self.fc2(features)

#         return features

In [ ]:
class RRNN(nn.Module):
    def __init__(
        self,
        n_vocab=n_vocab,
        hidden_dim=hidden_dim,
        embedding_dim=embedding_dim,
        dropout=0.1,
    ):
        super(RRNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=False)
        self.hidden_dim = hidden_dim
        self.out_features_num = n_vocab

        self.lstm_desc = nn.GRU(
            embedding_dim,
            hidden_dim,
            dropout=dropout,
            num_layers=1,
            bidirectional=False,
            batch_first=True,
        )

        self.pooling_result_dimension = self.hidden_dim * 1
        self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

        self.attention_descs = Attention(self.hidden_dim * 1, batch_first=True)
        self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
        self.dropout1 = nn.Dropout(dropout, inplace=True)
        self.fc1 = nn.Linear(self.dim_after_concat, self.embedding_dim)
        self.dropout2 = nn.Dropout(dropout, inplace=True)
        self.fc2 = nn.Linear(self.embedding_dim, self.out_features_num)

        self.fc2.weight = self.embeddings.weight

        # self.category_embedding = self._create_embedding_projection(cardinality=10, embed_size=6, padding_idx=None, scale_grad_by_freq=False)
        # self.subcategory_embedding = self._create_embedding_projection(cardinality=50, embed_size=14, padding_idx=None, scale_grad_by_freq=False)
        # self.region_embedding = self._create_embedding_projection(cardinality=85, embed_size=19, padding_idx=None, scale_grad_by_freq=False)

    def _create_embedding_projection(
        self,
        cardinality,
        embed_size,
        add_missing=True,
        padding_idx=0,
        scale_grad_by_freq=False,
    ):
        add_missing = add_missing * 1
        return nn.Embedding(
            num_embeddings=cardinality + add_missing,
            embedding_dim=embed_size,
            padding_idx=padding_idx,
            scale_grad_by_freq=scale_grad_by_freq,
        )

    def forward(self, aids, lens, targets=None):
        batch_size = aids.shape[0]

        # category_embed = self.category_embedding(category)
        # subcategory_embed = self.subcategory_embedding(subcategory)
        # region_embed = self.region_embedding(region)

        embedded_aids = self.embeddings(aids)
        pad_embed_pack = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_aids, list(lens), batch_first=True, enforce_sorted=False
        )
        states_aids, hh_aids = self.lstm_desc(pad_embed_pack)

        states_aids, lens_unpacked = torch.nn.utils.rnn.pad_packed_sequence(
            states_aids, batch_first=True
        )

        lengths_aids = [x_.size(1) for x_ in states_aids]
        atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
        atten_aids = atten_aids.view(batch_size, -1)

        features = torch.cat(
            [
                # subcategory_embed, category_embed, region_embed,
                atten_aids
            ],
            dim=-1,
        )
        features = self.bn1(features)
        features = self.dropout1(features)
        features = features.view(features.size(0), -1)
        features = self.fc1(features)
        features = self.dropout2(features)
        features = self.fc2(features)
        # features = F.normalize(features)

        # if targets is not None:
        #     embedded_targets = self.embeddings(targets)
        #     # print(embedded_targets.size())
        #     # print(features.size())
        #     features = torch.mm(embedded_targets, features.T)
        return features

In [ ]:
X_train.head()

In [ ]:
# X_val["val"].value_counts()

In [ ]:
X_val.head()

In [ ]:
X_train.shape, X_val.shape

In [ ]:
# TRAIN_BATCH_SIZE = 128
# INFER_BATCH_SIZE = 128

# dataset_train = Dataset_RNN(X_train, mode="train")
# dataset_local_valid = Dataset_RNN(X_val, mode="train")

# train_loader = torch.utils.data.DataLoader(
#     dataset_train,
#     batch_size=TRAIN_BATCH_SIZE,
#     shuffle=True,
#     num_workers=0,
#     drop_last=False,
# )
# local_valid_loader = torch.utils.data.DataLoader(
#     dataset_local_valid,
#     batch_size=INFER_BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     drop_last=False,
# )


# dataset_local_inference = Dataset_RNN(X_val, mode='test')
# local_inference_loader = torch.utils.data.DataLoader(
#     dataset_local_inference, batch_size=INFER_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False,
# )

# dataset_inference = Dataset_RNN(df_val, mode='test')
# inference_loader = torch.utils.data.DataLoader(
#     dataset_inference, batch_size=INFER_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False,
# )

In [ ]:
# %%time

# X_train["train"] = X_train["train"].progress_apply(lambda x: torch.tensor(x).long())
# X_train["val"] = X_train["val"].progress_apply(lambda x: torch.tensor(x).float())
# X_val["train"] = X_val["train"].progress_apply(lambda x: torch.tensor(x).long())
# X_val["val"] = X_val["val"].progress_apply(lambda x: torch.tensor(x).float())

In [ ]:
%%time
X_train = X_train.to_pandas()
X_val = X_val.to_pandas()

X_train_to_dataset = pd.DataFrame(X_train[config.col_train_seq].tolist(), index=X_train.index)
X_train_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_train_to_dataset.columns]
X_train_to_dataset[config.col_target] = X_train[config.col_target]

X_val_to_dataset = pd.DataFrame(X_val[config.col_train_seq].tolist(), index=X_val.index)
X_val_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_val_to_dataset.columns]
X_val_to_dataset[config.col_target] = X_val[config.col_target]

X_val_target_to_dataset = pd.DataFrame(X_val_to_dataset[config.col_target].tolist(), index=X_val_to_dataset.index)
X_val_target_to_dataset.columns = ['target' + '_' + str(pos) for pos in X_val_target_to_dataset.columns]
X_val_to_dataset[X_val_target_to_dataset.columns] = X_val_target_to_dataset[X_val_target_to_dataset.columns]
X_val_to_dataset.drop([config.col_target], axis=1, inplace=True)

X_train = pl.from_pandas(X_train)
X_val = pl.from_pandas(X_val)

cols_pos = [col for col in X_train_to_dataset.columns if "pos" in col]
cols_target_val = [col for col in X_val_to_dataset.columns if "target" in col]

In [ ]:
cols_target_val

In [ ]:
X_train_to_dataset.head()

In [ ]:
X_val_to_dataset.head()

In [ ]:
%%time

X_train_to_dataset.to_parquet('train.parquet', index=False)
X_val_to_dataset.to_parquet('valid.parquet', index=False)

In [ ]:
%%time

train_ds = Dataset_merlin('train.parquet')
train_dl_merlin = Loader(dataset=train_ds, batch_size=config.batch_size_train, shuffle=True)

valid_ds = Dataset_merlin('valid.parquet')
valid_dl_merlin = Loader(dataset=valid_ds, batch_size=config.batch_size_infer, shuffle=False)

In [ ]:
import math


def get_matrix_and_target_from_batch(
    batch, cols_pos=cols_pos, cols_target_val=cols_target_val, pad=pad, train=True
):
    batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
    for i in range(len(cols_pos)):
        batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
    batch_matrix = [
        torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
        for i in range(len(batch_matrix))
    ]
    lens = torch.tensor(list(map(len, batch_matrix)))

    batch_matrix = torch.nn.utils.rnn.pad_sequence(
        batch_matrix, batch_first=True, padding_value=pad
    )

    if train:
        target = batch[config.col_target]
    else:
        target = torch.zeros([len(batch[cols_target_val[0]]), len(cols_target_val)])
        for i in range(len(cols_target_val)):
            target[:, i] = torch.reshape(batch[cols_target_val[i]], (1, -1))
        target = [
            torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
            for i in range(len(target))
        ]
    return batch_matrix, target, lens

In [ ]:
%%time

for batch, _ in tqdm(train_dl_merlin):
    # batch_matrix, batch["val"], lens = get_matrix_and_target_from_batch(batch, cols_pos=cols_pos, pad=pad)
    continue
    # break
# 5 min

In [ ]:
# %%time

# batch_matrix = np.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
# for i in range(len(cols_pos)):
#     batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
# batch_matrix = [
#         torch.tensor([int(val) for val in batch_matrix[i] if not np.isnan(val)])
#         for i in range(len(batch_matrix))
#     ]

In [ ]:
# def get_matrix_and_target_from_batch(batch, cols_pos=cols_pos):
#     batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
#     for i in range(len(cols_pos)):
#         batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
#     # batch_target = torch.reshape(batch["val"], (1, -1))
#     return batch_matrix, batch["val"]

In [ ]:
DEBUG = False


def train_func(
    train_loader, model, criterion, opts: list, amp=False, num_batches_to_check=1000
):
    model.train()
    bar = tqdm(train_loader)
    losses = list()
    if amp:
        scaler = torch.cuda.amp.GradScaler()

    start_time = time.perf_counter()
    batches_passed = 0
    for batch_idx, (batch, _) in enumerate(bar):
        if num_batches_to_check:
            if batch_idx % num_batches_to_check == 0 and batch_idx > 0:
                batches_passed += num_batches_to_check
                end_time = time.perf_counter()
                total_time = end_time - start_time
                print(f"batches passed: {batches_passed}")
                print(
                    f"last {num_batches_to_check} batches took {total_time / 60 :.4f} minutes"
                )
                print()
                start_time = time.perf_counter()

        aids, targets, lens = get_matrix_and_target_from_batch(batch)
        aids = aids.to(device).long()
        lens = lens.to(device).long()
        # category = category.to(device).long()
        # subcategory = subcategory.to(device).long()
        # region = region.to(device).long()

        targets = targets.to(device).long()
        targets = targets.squeeze(1)

        if DEBUG and batch_idx == 2:
            print("Debug Mode. Only train on first 1 batches.")
            break

        if amp:
            with torch.cuda.amp.autocast():
                logits = model(aids, lens, targets)
                loss = criterion(logits, targets)
                scaler.scale(loss).backward()  # loss.backward()
                for optimizer in opts:
                    scaler.step(optimizer)  # optimizer.step()
                    scaler.update()
        else:
            logits = model(aids, lens, targets)
            # print(logits.size())
            loss = criterion(logits, targets)
            loss.backward()
        for optimizer in opts:
            optimizer.step()

        for optimizer in opts:
            optimizer.zero_grad()

        losses.append(loss.item())
        # smooth_loss = np.mean(losses[-30:])

        # bar.set_description(f"smooth_loss: {smooth_loss:.5f}")

    loss_train = np.mean(losses)
    print("loss_train:", loss_train)
    return loss_train


def do_inverse_transform(array, ordinal_encoder=ordinal_encoder):
    result = ordinal_encoder.inverse_transform(array.reshape(-1, 1)).reshape(1, -1)[0]
    result = [val for val in result if val is not None]
    return result


def flatten_list(l):
    return [item for sublist in l for item in sublist]


def valid_func(valid_loader, model, criterion):
    model.eval()

    bar = tqdm(valid_loader)
    PREDS = list()
    TARGETS = list()

    losses = list()
    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, targets, lens = get_matrix_and_target_from_batch(batch, train=False)
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            # category = category.to(device).long()
            # subcategory = subcategory.to(device).long()
            # region = region.to(device).long()
            # targets = targets.to(device).long()
            # targets = targets.squeeze(1)

            logits = model(aids, lens)
            # loss = criterion(logits, targets)
            # losses.append(loss.item())

            # PREDS += [torch.argmax(logits, 1).detach().cpu()]
            k = 20
            preds_cur = torch.topk(logits, k).indices.detach().cpu().numpy()
            # print(preds_cur.shape)
            # print(preds_cur)
            # preds_cur = [np.argpartition(-pred, k)[:k] for pred in preds_cur]
            # preds_cur = [
            #     do_inverse_transform(pred, ordinal_encoder=ordinal_encoder)
            #     for pred in preds_cur
            # ]
            PREDS.append(preds_cur)
            targets = [target.tolist() for target in targets]
            TARGETS.append(targets)

            # bar.set_description(f"loss: {loss.item():.5f}")

    # loss_valid = np.mean(losses)
    # print("loss_valid:", loss_valid)

    # PREDS = torch.cat(PREDS).cpu().tolist()
    PREDS = flatten_list(PREDS)
    TARGETS = flatten_list(TARGETS)
    # TARGETS = torch.cat(TARGETS).cpu().tolist()
    # print(len(PREDS), len(TARGETS))
    # print(TARGETS)
    # print(PREDS)
    df_val = pd.DataFrame({"idx": np.arange(len(TARGETS)), "preds": PREDS})
    df_val = df_val.explode("preds")

    df_val["preds"] = ordinal_encoder.inverse_transform(
        df_val["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_val = df_val.dropna()
    df_val["preds"] = df_val["preds"].astype(int)
    df_val = pl.from_pandas(df_val)
    df_val = (
        (
            df_val.groupby(["idx"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("idx")
        .to_pandas()
    )

    PREDS = df_val["preds"].tolist()
    PREDS = [list(pred) for pred in PREDS]

    metric_total = utils.recall_several_targets(PREDS, TARGETS)
    print("metric_total:", metric_total)

    return PREDS


# def inference_func(inference_loader, model):
#     model.eval()

#     bar = tqdm(inference_loader)
#     preds = list()

#     with torch.no_grad():
#         for batch_idx, (descs, category, subcategory, region) in enumerate(bar):
#             descs = descs.to(device).long()
#             category = category.to(device).long()
#             subcategory = subcategory.to(device).long()
#             region = region.to(device).long()

#             logits = model(descs, category, subcategory, region)

#             preds += [logits.detach().cpu()]

#     preds = torch.cat(preds).cpu().numpy()
#     preds = 1.0 / (1.0 + np.exp(-preds))
#     preds = np.concatenate(preds, axis=0)
#     return preds

In [ ]:
out_features_num = pad  # + 1
out_features_num

In [ ]:
model = RRNN(n_vocab, hidden_dim, embedding_dim)
model.to(device)

In [ ]:
params_sparsed = list()
params_not_sparsed = list()
cnt = 0
for param in model.parameters():
    if cnt == 0:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)
    cnt += 1

In [ ]:
params_sparsed[0].shape

In [ ]:
class MultipleOptimizer:
    def __init__(self, *optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for optimizer in self.optimizers:
            optimizer.zero_grad()

    def step(self):
        for optimizer in self.optimizers:
            optimizer.step()


class MultipleScheduler:
    def __init__(self, *schedulers):
        self.schedulers = schedulers

    def step(self):
        for scheduler in self.schedulers:
            scheduler.step()

In [ ]:
init_lr = 0.003
n_epochs = 3

criterion = nn.CrossEntropyLoss(ignore_index=pad)
optimizer = optim.Adam(model.parameters(), lr=init_lr)
optimizer_sparsed = optim.SparseAdam(params_sparsed, lr=init_lr)
optimizer_not_sparsed = optim.Adam(params_not_sparsed, lr=init_lr)
opts = [optimizer]  # , optimizer_sparsed, optimizer_not_sparsed
opt = MultipleOptimizer(optimizer_sparsed, optimizer_not_sparsed)

# scheduler_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler_not_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler = MultipleScheduler(scheduler_sparsed, scheduler_not_sparsed)

seed_everything(42)

In [ ]:
%%time

for epoch in tqdm(range(n_epochs), desc="epochs"):

    print(f"Epoch {epoch + 1} started")
    torch.cuda.empty_cache()

    # print("LR:", scheduler.schedulers[0].state_dict()["_last_lr"][0])
    loss_train = train_func(train_dl_merlin, model, criterion, opts)
    # scheduler.step()

    preds_local = valid_func(valid_dl_merlin, model, criterion)
    # X_val['preds'] = preds_local
    # aucs = list()
    # for category, cur_df in X_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('local mean_auc:', np.mean(aucs))

    # loss_valid, preds = valid_func(valid_loader, model, criterion)
    # df_val['preds'] = preds
    # aucs = list()
    # for category, cur_df in df_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('mean_auc:', np.mean(aucs))

    torch.save(
        model.state_dict(),
        f"_epoch{epoch + 1}.pth",
    )

    print("Checkpoint saved!")
    print()
    print("*" * 90)
    print()
    print()

# del model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# metric_total: 0.042083333333333334 1 ep
# metric_total: 0.06625 1 ep 5 iter
# metric_total: 0.059583333333333335 1 ep 10 iter
# metric_total: 0.0725 1 ep 15 iter

In [ ]:
X_val[config.col_target].value_counts().head(10)

In [ ]:
X_train["val"].value_counts().head(10)

In [ ]:
X_val = X_val.to_pandas()

In [ ]:
X_val["preds"] = preds_local
X_val["recall_row"] = X_val.progress_apply(
    lambda x: utils.recall_several_targets_one_row(
        x["preds"], x[config.col_target].tolist()
    ),
    axis=1,
)

In [ ]:
X_val.shape

In [ ]:
X_val.tail()

In [ ]:
stop

In [ ]:
X_val[X_val["recall_row"] == 1].shape

In [ ]:
X_val_true = X_val[X_val["recall_row"] == 1]

In [ ]:
pad

In [ ]:
X_val_true.head()

In [ ]:
X_train[X_train["val"] == pad].shape[0] / X_train.shape[0]

In [ ]:
X_val_true.iloc[0]["preds"]

In [ ]:
stop
# loss_train: 12.190348625183105

# loss_train: 9.231197555859884
# loss_valid: 20.559587955474854
# metric_total: 0.000999000999000999

In [ ]:
params_sparsed = list()
params_not_sparsed = list()
for param in model.parameters():
    if param.grad.is_sparse:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)

In [ ]:
params_sparsed

In [ ]:
df_train.groupby(["n_week"]).count().sort("n_week", reverse=False)

In [ ]:
df_train_ex = df_train.filter(pl.col("user") <= 10000)

In [ ]:
%%time

dict_train = dict()
dict_answers = dict()
dict_val_len = dict()
dict_user = dict()
key = 1

for df_user in tqdm(df_train.groupby(["user"], maintain_order=True)):
    df_weeks_count = df_user.groupby(["n_week"]).count().sort("n_week", reverse=False)
    if len(df_weeks_count) <= 1:
        continue
    user = df_user["user"][0]
    for i in range(len(df_weeks_count["n_week"]) - 1):
        n_week_cur = df_weeks_count["n_week"][i]
        val_len = 34 - n_week_cur
        dict_train[key] = df_user.filter(pl.col("n_week") == n_week_cur)
        dict_answers[key] = df_user.filter(pl.col("n_week") > n_week_next)
        dict_val_len[key] = val_len
        dict_user[key] = user
        key += 1

In [ ]:
dict_train[1]

In [ ]:
dict_answers[1]

In [ ]:
df_user.filter(pl.col("n_week") == n_week_cur)

In [ ]:
for i in range(len(df_weeks_count["n_week"]) - 1):
    n_week_cur = df_weeks_count["n_week"][i]
    n_week_next = df_weeks_count["n_week"][i + 1]
    print(n_week_cur, n_week_next)

In [ ]:
df_user

In [ ]:
df_train_ex

In [ ]:
df_train.tail(10)

In [ ]:
df_test.head()

In [ ]:
utils.plot_user_activity(user=12899774, df=df_train)

In [ ]:
df_train["type"].value_counts()

In [ ]:
df_train["date"].min(), df_train["date"].max()

In [ ]:
df_test["date"].min(), df_test["date"].max()

In [ ]:
%%time

df_train.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
%%time

df_test.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
df_train_n_session = (df_train.groupby("user")[["n_session"]].max() + 1).reset_index()
df_test_n_session = (df_test.groupby("user")[["n_session"]].max() + 1).reset_index()

In [ ]:
df_train_n_session["n_session"].value_counts()

In [ ]:
df_test_n_session["n_session"].value_counts()

In [ ]:
def plot_n_interactions_per_dayofweek(df):
    try:
        df_name = df.name
    except AttributeError:
        df_name = None

    title = "n_interactions per dayofweek"
    if df_name is not None:
        title += f" in {df_name}"

    plt.figure(figsize=(12, 7))
    plt.title(title)
    plt.xlabel("dayofweek")
    plt.ylabel("n_interactions")

    dict_type_inter_offset = dict()
    dict_type_inter_offset["clicks"] = -0.2
    dict_type_inter_offset["carts"] = 0.0
    dict_type_inter_offset["orders"] = 0.2

    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "g"
    dict_type_inter_color["orders"] = "r"

    for (dayofweek, type_inter), df_cur in df.groupby(["dayofweek", "type"]):
        ys = df_cur["ts"].count()
        plt.bar(
            dayofweek + dict_type_inter_offset[type_inter],
            ys,
            width=0.2,
            color=dict_type_inter_color[type_inter],
            align="center",
            # label=feature,
            # marker=",",
            # linestyle="dashed",
            alpha=0.6,
        )

In [ ]:
plot_n_interactions_per_dayofweek(df_train)

In [ ]:
plot_n_interactions_per_dayofweek(df_test)

In [ ]:
def plot_user_activity(user, df):
    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "orange"
    dict_type_inter_color["orders"] = "r"

    offset = 0.5
    df_cur = df[df["user"] == user]
    df_cur["date_day"] = df_cur["date_day"].astype(str)

    if df.name == "df_train":
        x = pd.date_range("2022-08-01", "2022-08-28")
    elif df.name == "df_test":
        x = pd.date_range("2022-08-29", "2022-09-04")
    x = [str(d).split(" ")[0] for d in x]
    map_dates = {x[i]: i for i in range(len(x))}
    points_sundays = [point for point in range(len(x)) if (point + 1) % 7 == 0]

    plt.figure(figsize=(12, 7))
    plt.xticks(range(len(x)), x, rotation=90)
    plt.yticks(range(25))

    for point in range(25):
        plt.axhline(y=point, color="gray", linestyle="-", alpha=0.1)
    for point in range(len(x)):
        plt.axvline(x=point + offset, color="gray", linestyle="-", alpha=0.1)
        plt.axvline(x=point - offset, color="gray", linestyle="-", alpha=0.1)
    for point in points_sundays:
        plt.axvline(x=point + offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset - 1, color="r", linestyle="-", alpha=0.3)

    for action, df_cur_action in df_cur.groupby("type"):
        xx = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        yy = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        plt.scatter(
            df_cur_action["date_day"].map(map_dates) + xx,
            df_cur_action["hour"] + yy,
            c=dict_type_inter_color[action],
            s=10,
            label=action,
        )
    plt.legend(
        loc="upper right", bbox_to_anchor=(1.1, 1.0), ncol=1, fancybox=True, shadow=True
    )

In [ ]:
plot_user_activity(user=2932269, df=df_train)

In [ ]:
plot_user_activity(user=12899780, df=df_test)